# 📊 Análisis Iterativo de Modelos de Series de Tiempo
## Riesgo País Argentino — VECM · VAR · ARDL

**Flujo de trabajo:**
1. Leer `Variables Regresivas/analisis_series_*.csv` (generado desde el visualizador)
2. Parsear el `CATALOG` de `visualizador_template.html` para obtener rutas y columnas de cada serie
3. Cargar datos desde `data/Variables Finales/` con la misma lógica que el visualizador (resample = media, deflactores idénticos)
4. Ejecutar pre-tests (ADF/KPSS, Granger, Johansen)
5. Estimar ARDL · VAR · VECM sobre **todas las combinaciones posibles** de variables explicativas
6. Visualizar coeficientes y criterios de información

---
**Solo editar la Sección 3 (CONFIGURACIÓN).**

## 1. Instalación e Imports

In [ ]:
import subprocess, sys, os, re

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for _pkg in ['statsmodels', 'plotly', 'scipy', 'scikit-learn', 'openpyxl']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        print(f'Instalando {_pkg}...')
        _install(_pkg)

import warnings
import glob
import random
from itertools import combinations
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import test_estacionariedad, test_causalidad_granger, resumen_estadistico

print('✅ Módulos cargados.')

## 2. Funciones Auxiliares

In [ ]:
# ── Directorios base (igual que generar_visualizador.py) ─────────────────────
VARS_DIR = (Path('..') / 'data' / 'Variables Finales').resolve()
RAW_DIR  = (Path('..') / 'data' / 'raw').resolve()

# ── 1. Parsear CATALOG desde visualizador_template.html ──────────────────────
def _parse_catalog():
    for _cand in [Path('visualizador_template.html'),
                  Path('notebooks') / 'visualizador_template.html']:
        if _cand.exists():
            html_path = _cand
            break
    else:
        raise FileNotFoundError('No se encontró visualizador_template.html')

    with open(html_path, 'r', encoding='utf-8') as f:
        content = f.read()

    cat = {}
    _SINGLE = re.compile(
        r"id\s*:\s*'([^']+)'.*?label\s*:\s*'([^']+)'.*?"
        r"file\s*:\s*'([^']+)'.*?dateCol\s*:\s*'([^']+)'.*?valCol\s*:\s*'([^']+)'"
    )
    _FREQ_RE = re.compile(r"freq\s*:\s*'([^']+)'")

    for line in content.splitlines():
        m = _SINGLE.search(line)
        if m:
            cid, label, file_, dc, vc = m.groups()
            fm = _FREQ_RE.search(line)
            freq = fm.group(1) if fm else 'D'
            cat[label] = {'id': cid, 'file': file_, 'dateCol': dc, 'valCol': vc, 'freq': freq}

    _MULTI = re.compile(
        r"id\s*:\s*'([^']+)'\s*,\s*label\s*:\s*'([^']+)'[^[]*?sources\s*:\s*\[(.*?)\]",
        re.DOTALL
    )
    _SRC = re.compile(
        r"freq\s*:\s*'([^']+)'[^}]*?file\s*:\s*'([^']+)'[^}]*?"
        r"dateCol\s*:\s*'([^']+)'[^}]*?valCol\s*:\s*'([^']+)'"
    )
    for m in _MULTI.finditer(content):
        cid, label, srcs_txt = m.groups()
        if label in cat:
            continue
        srcs = [{'freq': sm.group(1), 'file': sm.group(2),
                 'dateCol': sm.group(3), 'valCol': sm.group(4)}
                for sm in _SRC.finditer(srcs_txt)]
        if srcs:
            _PREF = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}
            best = sorted(srcs, key=lambda s: _PREF.get(s['freq'], 99))[0]
            cat[label] = {'id': cid, 'file': best['file'],
                          'dateCol': best['dateCol'], 'valCol': best['valCol'],
                          'freq': best['freq'], 'sources': srcs}
    return cat


CATALOG_VIZ = _parse_catalog()
print(f'✅ CATALOG parseado: {len(CATALOG_VIZ)} series')


# ── 2. Detección automática de frecuencia ─────────────────────────────────────
def detectar_frecuencia(df_csv):
    """
    Para cada serie del CSV busca la frecuencia más fina disponible en CATALOG.
    Retorna la más gruesa entre esas frecuencias finas (= mínima frecuencia común).
    Ejemplo: Serie A disponible en D y M → más fina = D.
             Serie B solo en Q          → más fina = Q.
             → usar Q (la más gruesa de {D, Q}).
    """
    _ORD = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}
    finest = []
    for _, row in df_csv.iterrows():
        for col in ['Serie A', 'Serie B']:
            nombre = str(row.get(col, '') or '').strip()
            if not nombre or nombre in ('', '—', '-', 'nan'):
                continue
            if nombre not in CATALOG_VIZ:
                continue
            meta = CATALOG_VIZ[nombre]
            if 'sources' in meta:
                src_freqs = [s['freq'] for s in meta['sources']]
                finest.append(min(src_freqs, key=lambda f: _ORD.get(f, 99)))
            else:
                finest.append(meta.get('freq', 'D'))

    if not finest:
        return 'M'
    return max(finest, key=lambda f: _ORD.get(f, -1))


# ── 3. Resampling (media por período, igual que el visualizador JS) ───────────
def _resample_mean(serie: pd.Series, to_freq: str) -> pd.Series:
    _MAP = {'D': 'D', 'W': 'W', 'M': 'ME', 'Q': 'QE', 'A': 'YE'}
    pf = _MAP.get(to_freq, to_freq)
    try:
        return serie.resample(pf).mean()
    except ValueError:
        _MAP_OLD = {'D': 'D', 'W': 'W', 'M': 'M', 'Q': 'Q', 'A': 'A'}
        return serie.resample(_MAP_OLD.get(to_freq, to_freq)).mean()


# ── 4. Carga genérica: busca en VARS_DIR primero, luego RAW_DIR ───────────────
def _buscar_archivo(*rutas_relativas):
    for ruta in rutas_relativas:
        for base in [VARS_DIR, RAW_DIR]:
            p = base / ruta
            if p.exists():
                return p
    return None


def _leer_serie_csv(path: Path, col_fecha: str, col_valor: str) -> pd.Series:
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    df['_f'] = pd.to_datetime(df[col_fecha], errors='coerce')
    df = df.dropna(subset=['_f']).set_index('_f').sort_index()
    return pd.to_numeric(df[col_valor], errors='coerce').dropna()


# ── 5. Deflactores (misma lógica que el visualizador JS) ─────────────────────
_cache_pbi = _cache_cer = _cache_tip = _cache_mep = None

def _get_pbi():
    global _cache_pbi
    if _cache_pbi is not None: return _cache_pbi
    p = _buscar_archivo('pbi_constante_2004.csv')
    if p is None: print('  ⚠️  pbi_constante_2004.csv no encontrado'); return None
    _cache_pbi = _leer_serie_csv(p, 'fecha', 'pbi'); return _cache_pbi

def _get_cer():
    global _cache_cer
    if _cache_cer is not None: return _cache_cer
    p = _buscar_archivo('cer.csv', 'bcra/cer.csv')
    if p is None: print('  ⚠️  cer.csv no encontrado'); return None
    _cache_cer = _leer_serie_csv(p, 'fecha', 'cer'); return _cache_cer

def _get_tip():
    global _cache_tip
    if _cache_tip is not None: return _cache_tip
    p = _buscar_archivo('tip_etf.csv', 'global/tip_etf.csv')
    if p is None: print('  ⚠️  tip_etf.csv no encontrado'); return None
    _cache_tip = _leer_serie_csv(p, 'fecha', 'tip_close'); return _cache_tip

def _get_mep():
    global _cache_mep
    if _cache_mep is not None: return _cache_mep
    p = _buscar_archivo('brecha_cambiaria.csv')
    if p is None: print('  ⚠️  brecha_cambiaria.csv no encontrado'); return None
    _cache_mep = _leer_serie_csv(p, 'fecha', 'mep'); return _cache_mep


def _aplicar_deflactor(serie: pd.Series, deflactor_name: str,
                        frecuencia: str) -> pd.Series:
    if not deflactor_name or str(deflactor_name).strip() in ('', '—', '-', 'nan'):
        return serie
    for d in str(deflactor_name).split(','):
        d = d.strip().upper()
        if not d or d in ('—', '-'): continue
        if d == 'PBI':
            raw = _get_pbi()
            if raw is None: continue
            al = _resample_mean(raw, frecuencia).reindex(serie.index, method='ffill')
            fp = al.dropna().iloc[0] if al.dropna().size else 1.0
            serie = serie * fp / al
        elif d == 'CER':
            raw = _get_cer()
            if raw is None: continue
            al = _resample_mean(raw, frecuencia).reindex(serie.index, method='ffill')
            lc = al.dropna().iloc[-1] if al.dropna().size else 1.0
            serie = serie / al * lc
        elif d in ('TIP', 'IPC'):
            raw = _get_tip()
            if raw is None: continue
            al = _resample_mean(raw, frecuencia).reindex(serie.index, method='ffill')
            lt = al.dropna().iloc[-1] if al.dropna().size else 1.0
            serie = serie / al * lt
        elif d == 'MEP':
            raw = _get_mep()
            if raw is None: continue
            al = _resample_mean(raw, frecuencia).reindex(serie.index, method='ffill')
            serie = serie / al
        else:
            print(f'    ⚠️  Deflactor no reconocido: "{d}"')
    return serie


# ── 6. Carga de serie desde CATALOG ──────────────────────────────────────────
def _cargar_serie_catalog(label: str, frecuencia: str,
                           fecha_inicio: str = None, fecha_fin: str = None):
    if label not in CATALOG_VIZ:
        print(f'  ⚠️  "{label}" no encontrado en CATALOG.')
        return None
    meta = CATALOG_VIZ[label]
    if 'sources' in meta:
        srcs = meta['sources']
        exact = [s for s in srcs if s['freq'] == frecuencia]
        _PREF = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}
        src = exact[0] if exact else sorted(srcs, key=lambda s: _PREF.get(s['freq'], 99))[0]
        file_, dc, vc = src['file'], src['dateCol'], src['valCol']
    else:
        file_, dc, vc = meta['file'], meta['dateCol'], meta['valCol']

    file_path = VARS_DIR / file_
    if not file_path.exists():
        print(f'  ⚠️  Archivo no encontrado: {file_path}')
        return None

    if str(file_).endswith(('.xlsx', '.xls')):
        sheet = 'Unificado' if 'fiscal' in str(file_).lower() else 0
        df = pd.read_excel(file_path, sheet_name=sheet)
    else:
        df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.str.strip()

    if dc not in df.columns:
        print(f'  ⚠️  Columna "{dc}" no hallada en {file_}. Cols: {list(df.columns[:6])}')
        return None
    if vc not in df.columns:
        print(f'  ⚠️  Columna "{vc}" no hallada en {file_}. Cols: {list(df.columns[:8])}')
        return None

    df['_f'] = pd.to_datetime(df[dc], errors='coerce')
    df = df.dropna(subset=['_f']).set_index('_f').sort_index()
    serie = pd.to_numeric(df[vc], errors='coerce').dropna()
    serie.name = label
    if fecha_inicio: serie = serie[serie.index >= pd.Timestamp(fecha_inicio)]
    if fecha_fin:    serie = serie[serie.index <= pd.Timestamp(fecha_fin)]
    return _resample_mean(serie, frecuencia).dropna()


# ── 7. Construcción de variable desde fila CSV ────────────────────────────────
def construir_variable(row: pd.Series, frecuencia: str,
                        fecha_inicio: str, fecha_fin: str):
    nombre_a  = str(row.get('Serie A', '') or '').strip()
    defl_a    = str(row.get('Deflactor A', '') or '').strip()
    nombre_b  = str(row.get('Serie B', '') or '').strip()
    defl_b    = str(row.get('Deflactor B', '') or '').strip()
    operacion = str(row.get('Operación', 'Solo A') or 'Solo A').strip()
    log_flag  = str(row.get('Log', 'No') or 'No').strip().lower() not in ('no', '0', 'false', '')
    n_diff    = int(float(str(row.get('Dif.', 0) or 0) or 0))

    defl_label = f'/{defl_a}' if defl_a and defl_a not in ('', '—', '-', 'nan') else ''
    b_label    = (f' {operacion} {nombre_b}'
                  if operacion != 'Solo A' and nombre_b not in ('', '—', '-', 'nan') else '')
    etiqueta   = f'{nombre_a}{defl_label}{b_label}'
    if log_flag: etiqueta = f'log({etiqueta})'
    if n_diff > 0: etiqueta = ('Δ' * n_diff) + etiqueta

    sa = _cargar_serie_catalog(nombre_a, frecuencia, fecha_inicio, fecha_fin)
    if sa is None: return None, etiqueta
    sa = _aplicar_deflactor(sa, defl_a, frecuencia)
    resultado = sa.copy()

    if operacion != 'Solo A' and nombre_b not in ('', '—', '-', 'nan'):
        sb = _cargar_serie_catalog(nombre_b, frecuencia, fecha_inicio, fecha_fin)
        if sb is not None:
            sb = _aplicar_deflactor(sb, defl_b, frecuencia)
            sb_al = sb.reindex(sa.index, method='ffill')
            op = operacion.replace(' ', '')
            if op in ('A−B', 'A-B'):         resultado = sa - sb_al
            elif op == 'A/B':                resultado = sa / sb_al
            elif op in ('(A−B)/B','(A-B)/B'): resultado = (sa - sb_al) / sb_al

    if log_flag: resultado = np.log(resultado.replace(0, np.nan))
    for _ in range(n_diff): resultado = resultado.diff()
    resultado = resultado.dropna()
    resultado.name = etiqueta
    return resultado, etiqueta


print('✅ Funciones auxiliares listas.')
print(f'   Series:      {VARS_DIR}')
print(f'   Deflactores: VARS_DIR → RAW_DIR (fallback)')

## 3. ══════════ CONFIGURACIÓN ══════════
> **Esta es la única celda que necesitás editar.**

In [ ]:
# ── Archivo CSV ───────────────────────────────────────────────────────────────
# El visualizador descarga el archivo a la carpeta del navegador.
# Copiarlo a: notebooks/Variables Regresivas/
# Dejar None para que tome automáticamente el más reciente.
# Para un archivo específico usar solo el nombre: 'analisis_series_2026-05-06.csv'
CSV_ANALISIS = None   # None = el más reciente en notebooks/Variables Regresivas/

# ── Variable dependiente ──────────────────────────────────────────────────────
# Número de fila en el CSV (1-indexed) o texto que aparezca en "Serie A"
VARIABLE_DEPENDIENTE = 2   # Fila 2 = EMBI+ ARG − EMBI+ LATAM

# ── Frecuencia ────────────────────────────────────────────────────────────────
# 'auto' = detecta la frecuencia más gruesa disponible entre todas las series
# Alternativas explícitas: 'D' 'W' 'M' 'Q' 'A'
FRECUENCIA = 'auto'

# ── Fechas ────────────────────────────────────────────────────────────────────
FECHA_INICIO = '2007-01-01'
FECHA_FIN    = '2024-12-31'

# ── Lags máximos ──────────────────────────────────────────────────────────────
MAX_LAGS_DEP = 4   # rezagos Y en ARDL
MAX_LAGS_IND = 4   # rezagos X en ARDL
MAX_LAGS_VAR = 8   # selección de rezagos en VAR/VECM

# ── Combinaciones ─────────────────────────────────────────────────────────────
# None = TODAS las combinaciones posibles de TODOS los tamaños
MAX_VARS_POR_MODELO = None   # None = sin límite (todas las variables)
MAX_COMBINACIONES   = None   # None = sin límite (todas las combinaciones)
SEMILLA             = 42

# ── Criterios de información ──────────────────────────────────────────────────
# Se prueban TODOS. Para ARDL y VAR se seleccionan rezagos con cada criterio
# y se guardan los 3 métricas (AIC, BIC, HQ) de cada estimación.
CRITERIOS_SEL = ['aic', 'bic', 'hqic']

# ── Nivel de significancia ────────────────────────────────────────────────────
NIVEL_SIG = 0.05

# ── Modelos a correr ──────────────────────────────────────────────────────────
CORRER_ARDL = True
CORRER_VAR  = True
CORRER_VECM = True   # solo cuando Johansen detecta cointegración

print('✅ Configuración guardada.')

## 4. Carga y Construcción del Panel

In [ ]:
random.seed(SEMILLA)
np.random.seed(SEMILLA)

# 1. Encontrar CSV ─────────────────────────────────────────────────────────────
_csv_dir = Path('Variables Regresivas')
if CSV_ANALISIS:
    _csv_path = _csv_dir / CSV_ANALISIS if not Path(CSV_ANALISIS).is_absolute() else Path(CSV_ANALISIS)
else:
    _candidates = sorted(glob.glob(str(_csv_dir / 'analisis_series_*.csv')))
    if not _candidates:
        raise FileNotFoundError(f'No encontré analisis_series_*.csv en {_csv_dir.resolve()}')
    _csv_path = Path(_candidates[-1])

print(f'📁 CSV: {_csv_path.name}')

# 2. Leer CSV ──────────────────────────────────────────────────────────────────
df_csv = pd.read_csv(_csv_path)
df_csv.columns = df_csv.columns.str.strip()
for _c in ['Deflactor A', 'Deflactor B', 'Serie B', 'Operación', 'Log']:
    if _c in df_csv.columns:
        df_csv[_c] = df_csv[_c].fillna('—').astype(str).str.strip()
if 'Dif.' in df_csv.columns:
    df_csv['Dif.'] = (pd.to_numeric(
        df_csv['Dif.'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0).astype(int))

print(f'   {len(df_csv)} variables en el CSV')
display(df_csv[['#','Serie A','Deflactor A','Serie B','Operación','Log','Dif.']].to_string())

# 3. Detectar frecuencia ───────────────────────────────────────────────────────
if FRECUENCIA == 'auto':
    FRECUENCIA_USADA = detectar_frecuencia(df_csv)
    print(f'\n🔍 Frecuencia auto-detectada: {FRECUENCIA_USADA}')
else:
    FRECUENCIA_USADA = FRECUENCIA
    print(f'\n📅 Frecuencia configurada: {FRECUENCIA_USADA}')

# 4. Identificar variable dependiente ──────────────────────────────────────────
if isinstance(VARIABLE_DEPENDIENTE, int):
    _idx_dep = VARIABLE_DEPENDIENTE - 1
else:
    _mask = df_csv['Serie A'].str.contains(str(VARIABLE_DEPENDIENTE), na=False)
    if not _mask.any():
        raise ValueError(f'Variable dependiente "{VARIABLE_DEPENDIENTE}" no encontrada en el CSV')
    _idx_dep = _mask.idxmax()

df_csv['_es_dep'] = False
df_csv.loc[_idx_dep, '_es_dep'] = True

# 5. Construir series ──────────────────────────────────────────────────────────
print(f'\n⏳ Cargando series ({FECHA_INICIO} → {FECHA_FIN})...')
_series  = {}
_eti_map = {}

for _i, _row in df_csv.iterrows():
    try:
        _s, _eti = construir_variable(_row, FRECUENCIA_USADA, FECHA_INICIO, FECHA_FIN)
        if _s is not None and len(_s) >= 20:
            _series[_eti]  = _s
            _eti_map[_i]   = _eti
            _tag = '(Y)' if _i == _idx_dep else '(X)'
            print(f'  ✅ {_tag} [{_i+1}] {_eti[:70]}  →  {len(_s)} obs')
        else:
            print(f'  ❌ [{_i+1}] {_row["Serie A"]}: no cargó o < 20 obs')
    except Exception as _e:
        print(f'  ❌ [{_i+1}] {_row["Serie A"]}: {_e}')

if not _series:
    raise RuntimeError('No se cargó ninguna serie.')

# 6. Panel unificado ───────────────────────────────────────────────────────────
df_panel = pd.concat(list(_series.values()), axis=1).dropna()
print(f'\n✅ Panel: {df_panel.shape[0]} obs × {df_panel.shape[1]} vars')
print(f'   Período: {df_panel.index.min().date()} → {df_panel.index.max().date()}')

ETI_DEP  = _eti_map.get(_idx_dep)
ETI_EXPL = [_eti_map[i] for i in _eti_map if i != _idx_dep and _eti_map[i] in df_panel.columns]

if ETI_DEP not in df_panel.columns:
    raise RuntimeError(f'Variable dependiente "{ETI_DEP}" no disponible en el panel.')

print(f'\n  Y = {ETI_DEP}')
print(f'  X = {ETI_EXPL}')

# 7. Nombres cortos (evita problemas con caracteres especiales en statsmodels) ──
_all_cols = [ETI_DEP] + ETI_EXPL
COL_SAFE  = {c: f'v{i:02d}' for i, c in enumerate(_all_cols)}
COL_LABEL = {v: k for k, v in COL_SAFE.items()}

df_safe = df_panel[_all_cols].rename(columns=COL_SAFE)
Y_SAFE  = COL_SAFE[ETI_DEP]
X_SAFE  = [COL_SAFE[e] for e in ETI_EXPL]

print('\n  Nombres cortos:')
for _s, _l in COL_LABEL.items():
    print(f'    {_s} → {_l}')

## 5. Estadísticos Descriptivos

In [ ]:
display(resumen_estadistico(df_panel).round(4))

# Gráfico de series
n_series = len(df_panel.columns)
fig = make_subplots(rows=n_series, cols=1, shared_xaxes=True,
                    subplot_titles=list(df_panel.columns),
                    vertical_spacing=0.04)
for i, col in enumerate(df_panel.columns):
    color = '#d62728' if col == ETI_DEP else '#1f77b4'
    fig.add_trace(
        go.Scatter(x=df_panel.index, y=df_panel[col], name=col,
                   line=dict(color=color, width=1.5), showlegend=False),
        row=i+1, col=1
    )
fig.update_layout(height=220*n_series, template='plotly_white',
                  title_text='Series del panel (rojo = variable dependiente)')
fig.show()

## 6. Tests Previos

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# ── 6.1 Estacionariedad ADF + KPSS ────────────────────────────────────────────
print('═'*70)
print('6.1  ESTACIONARIEDAD (ADF + KPSS)')
print('═'*70)
_est_rows = []
for _col in df_panel.columns:
    _r = test_estacionariedad(df_panel[_col], _col)
    _est_rows.append(_r)
df_est = pd.DataFrame(_est_rows).set_index('Variable')
display(df_est[['ADF_stat','ADF_pvalue','ADF_estacionaria',
                'KPSS_stat','KPSS_pvalue','KPSS_estacionaria','Conclusión']])

# ── 6.2 Causalidad de Granger ─────────────────────────────────────────────────
print('\n' + '═'*70)
print('6.2  CAUSALIDAD DE GRANGER  (X → Y)')
print('═'*70)
_gr_rows = []
for _x in ETI_EXPL:
    if _x in df_panel.columns:
        try:
            _r = test_causalidad_granger(df_panel[[ETI_DEP, _x]], ETI_DEP, _x, max_lags=12)
            _gr_rows.append({
                'Variable X': _x[:60],
                'Mejor Lag':  _r['Mejor_Lag'],
                'P-Value':    round(_r['P_Value'], 4),
                'Causa Granger': '✅ Sí' if _r['Causalidad_Granger'] else '❌ No'
            })
        except Exception as _e:
            print(f'  ⚠️ {_x[:50]}: {_e}')
if _gr_rows:
    display(pd.DataFrame(_gr_rows).set_index('Variable X'))

# ── 6.3 Johansen (todas las variables juntas) ─────────────────────────────────
print('\n' + '═'*70)
print('6.3  COINTEGRACIÓN DE JOHANSEN (todas las variables)')
print('═'*70)
_N_COINT_GLOBAL = 0
try:
    _data_j = df_safe[[Y_SAFE] + X_SAFE].dropna()
    _joh = coint_johansen(_data_j, det_order=0, k_ar_diff=2)
    print(f'  Obs: {len(_data_j)} | Variables: {len(_data_j.columns)}')
    print(f'  {"r":>4} | {"Trace":>10} {"CV5%":>10} {"Sig":>5} | {"MaxEig":>10} {"CV5%":>10} {"Sig":>5}')
    print(f'  {"-"*60}')
    for _r in range(len(_joh.lr1)):
        _st = '✅' if _joh.lr1[_r] > _joh.cvt[_r, 1] else '  '
        _sm = '✅' if _joh.lr2[_r] > _joh.cvm[_r, 1] else '  '
        print(f'  {_r:>4} | {_joh.lr1[_r]:>10.3f} {_joh.cvt[_r,1]:>10.3f} {_st:>5} | '
              f'{_joh.lr2[_r]:>10.3f} {_joh.cvm[_r,1]:>10.3f} {_sm:>5}')
    _N_COINT_GLOBAL = int(sum(_joh.lr1 > _joh.cvt[:, 1]))
    print(f'\n  → Relaciones de cointegración (Trace, 5%): {_N_COINT_GLOBAL}')
except Exception as _e:
    print(f'  ❌ {_e}')

print('\n✅ Tests previos completados.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.4  Análisis BJ: CCF pairwise + propuesta de rezagos para ARDL
# Resultado: _BJ_PROPOSED_LAGS {x_var → [lags]}, _BJ_AR_Y (orden AR de Y)
# Estos se usan automáticamente en extraer_ardl() de la sección 7.
# ═══════════════════════════════════════════════════════════════════════════
try:
    from pmdarima import auto_arima as _auto_arima_bj
except ImportError:
    import subprocess as _sp_bj, sys as _sys_bj
    _sp_bj.check_call([_sys_bj.executable, '-m', 'pip', 'install', 'pmdarima', '-q'])
    from pmdarima import auto_arima as _auto_arima_bj

from statsmodels.tsa.stattools import ccf as _ccf_fn_bj

BJ_MAX_LAGS_CCF = 12   # rezagos máximos en el CCF
BJ_MAX_AR       = 4    # max p/q en auto_arima
BJ_Z_CI         = 1.96 # banda de confianza CCF (95%)

_y_bj = df_safe[Y_SAFE].dropna()

# Blanquear Y una sola vez (ARMA de Y)
_fit_y_bj = _auto_arima_bj(
    _y_bj.values, information_criterion='aic',
    stepwise=True, seasonal=False, d=0,
    max_p=BJ_MAX_AR, max_q=BJ_MAX_AR,
    suppress_warnings=True, error_action='ignore',
)
_p_y_bj, _, _q_y_bj = _fit_y_bj.order
_y_white_bj = pd.Series(
    _fit_y_bj.resid(),
    index=_y_bj.index[-len(_fit_y_bj.resid()):],
    name='y_w',
)
print(f'6.4  Y blanqueada: ARMA({_p_y_bj},{_q_y_bj})')

_BJ_PROPOSED_LAGS = {}   # {x_safe → lista de rezagos propuestos}
_bj_ar_y_estimates = [] # AR(Y) estimados por variable
_bj_ccf_store = {}       # para graficar despues

import statsmodels.api as _sm_bj

for _xvar_bj in X_SAFE:
    _xlabel_bj = COL_LABEL[_xvar_bj]
    _xs = df_safe[_xvar_bj].dropna()
    _df_xy_bj = pd.concat([_y_bj.rename('y'), _xs.rename('x')], axis=1).dropna()
    if len(_df_xy_bj) < 30:
        _BJ_PROPOSED_LAGS[_xvar_bj] = [0, 1]
        continue

    try:
        # 1.1 ARMA de X
        _fit_x_bj = _auto_arima_bj(
            _df_xy_bj['x'].values, information_criterion='aic',
            stepwise=True, seasonal=False, d=0,
            max_p=BJ_MAX_AR, max_q=BJ_MAX_AR,
            suppress_warnings=True, error_action='ignore',
        )
        _p_x_bj, _, _q_x_bj = _fit_x_bj.order
        _x_w_arr = _fit_x_bj.resid()
        _x_white_bj = pd.Series(_x_w_arr,
                                  index=_df_xy_bj['x'].index[-len(_x_w_arr):],
                                  name='x_w')

        # 1.2 CCF entre X blanqueada e Y blanqueada
        _df_w_bj = pd.concat([_x_white_bj, _y_white_bj], axis=1).dropna()
        _n_w_bj  = len(_df_w_bj)
        _ci_bj   = BJ_Z_CI / np.sqrt(_n_w_bj)
        _ccf_v_bj = _ccf_fn_bj(
            _df_w_bj['x_w'].values, _df_w_bj['y_w'].values,
            nlags=BJ_MAX_LAGS_CCF, alpha=None,
        )

        _sig_bj   = [int(l) for l in range(len(_ccf_v_bj)) if abs(_ccf_v_bj[l]) > _ci_bj]
        _prop_bj  = sorted({l for l in _sig_bj if 0 <= l <= MAX_LAGS_IND})
        if not _prop_bj:
            _prop_bj = [0, 1]

        # 1.3 OLS con lags propuestos → ARMA de residuos → AR(Y)
        _df_ols_bj = _df_xy_bj.copy()
        for _lag_bj in _prop_bj:
            _df_ols_bj[f'xL{_lag_bj}'] = _df_ols_bj['x'].shift(_lag_bj)
        _df_ols_bj = _df_ols_bj.dropna()
        _Xm_bj = _sm_bj.add_constant(_df_ols_bj[[f'xL{l}' for l in _prop_bj]])
        _resid_ols_bj = _sm_bj.OLS(_df_ols_bj['y'], _Xm_bj).fit().resid
        _fit_r_bj = _auto_arima_bj(
            _resid_ols_bj.values, information_criterion='aic',
            stepwise=True, seasonal=False, d=0,
            max_p=BJ_MAX_AR, max_q=BJ_MAX_AR,
            suppress_warnings=True, error_action='ignore',
        )
        _p_r_bj, _, _ = _fit_r_bj.order
        _bj_ar_y_estimates.append(max(_p_r_bj, 1))

        _BJ_PROPOSED_LAGS[_xvar_bj] = _prop_bj
        _bj_ccf_store[_xvar_bj] = {
            'label'  : _xlabel_bj,
            'arma_x' : (_p_x_bj, _q_x_bj),
            'ccf'    : _ccf_v_bj,
            'ci'     : _ci_bj,
            'prop'   : _prop_bj,
        }
        print(f'  {_xlabel_bj[:55]:<55}  ARMA_X=({_p_x_bj},{_q_x_bj})  '
              f'lags={_prop_bj}  AR_Y={max(_p_r_bj,1)}')

    except Exception as _e_bj:
        _BJ_PROPOSED_LAGS[_xvar_bj] = [0, 1]
        print(f'  WARN {COL_LABEL[_xvar_bj][:40]}: {_e_bj}')

# AR(Y) global: mediana de los estimados individuales
_BJ_AR_Y = int(round(np.median(_bj_ar_y_estimates))) if _bj_ar_y_estimates else 1
print(f'\n  AR(Y) global propuesto: {_BJ_AR_Y} '
      f'(mediana de {_bj_ar_y_estimates})')
print(f'\n Lags propuestos (resumen):')
for _k, _v in _BJ_PROPOSED_LAGS.items():
    print(f'   {COL_LABEL[_k][:45]:<45} -> {_v}')

# ── 1.6a  Correlograma Cruzado: X blanqueada vs Y blanqueada ─────────────────
print('\n6.4  Correlograma Cruzado (CCF) — X blanqueada vs Y blanqueada')
_nv_ccf   = len(_bj_ccf_store)
_nc_ccf   = 2
_nr_ccf   = (_nv_ccf + _nc_ccf - 1) // _nc_ccf

_titles_ccf = [
    (f"{v['label'][:38]}<br>"
     f"<sup>ARMA_X=({v['arma_x'][0]},{v['arma_x'][1]}) | "
     f"Y: ARMA({_p_y_bj},{_q_y_bj})</sup>")
    for v in _bj_ccf_store.values()
]

_fig_ccf_bj = make_subplots(
    rows=_nr_ccf, cols=_nc_ccf,
    subplot_titles=_titles_ccf,
    vertical_spacing=0.10,
    horizontal_spacing=0.10,
)

for _idx_ccf, (_xvar_ccf, _res_ccf) in enumerate(_bj_ccf_store.items()):
    _r_ccf = _idx_ccf // _nc_ccf + 1
    _c_ccf = _idx_ccf % _nc_ccf + 1
    _cv    = _res_ccf['ccf']
    _ci_v  = _res_ccf['ci']
    _prop_v = set(_res_ccf['prop'])
    _lgs   = list(range(len(_cv)))

    _clrs_ccf = [
        '#d62728' if l in _prop_v else
        '#1565C0' if abs(_cv[l]) > _ci_v else
        '#BDBDBD'
        for l in _lgs
    ]

    _fig_ccf_bj.add_trace(
        go.Bar(
            x=_lgs, y=[float(v) for v in _cv],
            marker_color=_clrs_ccf,
            name=_res_ccf['label'],
            showlegend=False,
            hovertemplate='Lag %{x}: CCF=%{y:.4f}<extra></extra>',
        ),
        row=_r_ccf, col=_c_ccf,
    )
    _fig_ccf_bj.add_hline(y=_ci_v,  line=dict(color='red', dash='dash', width=1.5),
                           row=_r_ccf, col=_c_ccf)
    _fig_ccf_bj.add_hline(y=-_ci_v, line=dict(color='red', dash='dash', width=1.5),
                           row=_r_ccf, col=_c_ccf)
    _fig_ccf_bj.add_hline(y=0, line=dict(color='black', width=0.5),
                           row=_r_ccf, col=_c_ccf)

    _prop_str_ccf = str(sorted(_prop_v))
    _fig_ccf_bj.update_xaxes(
        title_text=f'Rezago | Propuestos: {_prop_str_ccf}',
        dtick=1, row=_r_ccf, col=_c_ccf,
    )
    _fig_ccf_bj.update_yaxes(title_text='CCF', row=_r_ccf, col=_c_ccf)

_fig_ccf_bj.update_layout(
    height=380 * _nr_ccf,
    template='plotly_white',
    title=(
        '6.4 Paso 1.2 — Correlograma Cruzado (CCF): X blanqueada vs Y blanqueada<br>'
        '<sup>'
        'Rojo = lag propuesto para ARDL | '
        'Azul = significativo (|CCF|>1.96/sqrt(n)) | '
        'Gris = no significativo | '
        'Bandas punteadas = +/-1.96/sqrt(n)'
        '</sup>'
    ),
)
_fig_ccf_bj.show()
print('OK  _BJ_PROPOSED_LAGS y _BJ_AR_Y listos para extraer_ardl()')


## 7. Funciones de Extracción de Coeficientes

In [ ]:
def _hqic(llf, k, n):
    """Hannan-Quinn information criterion."""
    return -2 * llf + 2 * k * np.log(np.log(max(n, 3)))


def extraer_ardl(df_m, y_var, x_vars, max_lags_dep, max_lags_ind, criterio):
    """
    Estima ARDL con metodologia Box-Jenkins.
    Usa _BJ_PROPOSED_LAGS (de la celda 6.4) para los rezagos de cada X
    y _BJ_AR_Y para el orden AR de Y.
    Si esas variables no existen, cae en ardl_select_order como fallback.

    'coef' = suma de todos los rezagos de X (efecto acumulado de CP).
    'pval' = p-valor del rezago mas significativo.
    """
    from statsmodels.tsa.ardl import ARDL as _ARDL_direct

    data = df_m[[y_var] + x_vars].dropna()
    y    = data[y_var]
    Xd   = data[x_vars]

    # ── Ruta BJ: lags propuestos por CCF ─────────────────────────────────
    _bj_lags_ok = ('_BJ_PROPOSED_LAGS' in dir() or '_BJ_PROPOSED_LAGS' in globals())
    if _bj_lags_ok:
        try:
            _bj_lp = globals()['_BJ_PROPOSED_LAGS']
            _bj_ary = int(globals().get('_BJ_AR_Y', 1))
            _order_dict = {
                x: max(_bj_lp.get(x, [0, 1])) for x in x_vars
            }
            res = _ARDL_direct(y, lags=_bj_ary,
                                exog=Xd, order=_order_dict,
                                trend='c').fit()
        except Exception:
            _bj_lags_ok = False

    # ── Fallback: ardl_select_order ───────────────────────────────────────
    if not _bj_lags_ok:
        from statsmodels.tsa.ardl import ardl_select_order
        sel = ardl_select_order(y, max_lags_dep, Xd, max_lags_ind,
                                ic=criterio, trend='c')
        res = sel.model.fit()

    n   = res.nobs
    k   = len(res.params)
    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(getattr(res, 'hqic', _hqic(res.llf, k, n)))

    coefs = {}
    for x in x_vars:
        # Formato statsmodels ARDL: 'x.L0', 'x.L1', ... o 'x', 'L1.x', ...
        px = [p for p in res.params.index
              if p.startswith(f'{x}.L') or p == x]
        if not px:  # fallback para formato ardl_select_order
            px = [p for p in res.params.index
                  if p == x or (p.startswith('L') and p.split('.', 1)[-1] == x)]
        if px:
            coefs[x] = {
                'coef': float(res.params[px].sum()),
                'pval': float(res.pvalues[px].min())
            }
        else:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'coefs': coefs, 'nobs': int(n)}


def extraer_var(df_m, y_var, x_vars, todas_vars, max_lags, criterio):
    """
    Estima VAR y extrae coeficiente y p-valor de X en la ecuacion de Y.
    Usa extraccion posicional (robusto ante nombres con caracteres especiales).
    """
    from statsmodels.tsa.api import VAR

    data = df_m[todas_vars].dropna()
    model = VAR(data)
    lag_sel = model.select_order(maxlags=max_lags)
    k_ar = max(1, lag_sel.selected_orders.get(criterio, 1))
    res = model.fit(k_ar)

    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(res.hqic)

    params_df = res.params
    pvals_df  = res.pvalues

    k_endog = len(todas_vars)
    var_pos = {v: i for i, v in enumerate(todas_vars)}

    coefs = {}
    if y_var in params_df.columns:
        yp = params_df[y_var]
        yv = pvals_df[y_var]
        for x in x_vars:
            xpos = var_pos.get(x)
            if xpos is None:
                coefs[x] = {'coef': np.nan, 'pval': np.nan}
                continue
            idx = [lag * k_endog + xpos for lag in range(k_ar)
                   if lag * k_endog + xpos < len(yp)]
            if idx:
                coefs[x] = {
                    'coef': float(yp.iloc[idx].sum()),
                    'pval': float(yv.iloc[idx].min())
                }
            else:
                coefs[x] = {'coef': np.nan, 'pval': np.nan}
    else:
        for x in x_vars:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'coefs': coefs,
            'k_ar': k_ar, 'nobs': int(res.nobs)}


def extraer_vecm(df_m, y_var, x_vars, todas_vars, k_ar_diff, det_order):
    """
    Estima VECM (si hay cointegracion) y extrae coefs de corto plazo (gamma)
    de la ecuacion Y para cada X.
    """
    from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
    from scipy.stats import t as t_dist

    data = df_m[todas_vars].dropna()
    joh  = coint_johansen(data, det_order=det_order, k_ar_diff=k_ar_diff)
    n_ci = int(sum(joh.lr1 > joh.cvt[:, 1]))

    if n_ci == 0:
        return None

    res = VECM(data, k_ar_diff=k_ar_diff, coint_rank=n_ci,
               deterministic='ci').fit()

    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(getattr(res, 'hqic',
                         _hqic(res.llf if hasattr(res, 'llf') else
                               -res.bic * res.nobs / 2,
                               len(todas_vars) * n_ci, res.nobs)))

    all_list = list(todas_vars)
    y_idx    = all_list.index(y_var)
    k_vars   = len(all_list)
    gamma    = res.gamma
    gamma_se = getattr(res, 'gamma_se', None)
    n_short  = k_ar_diff - 1

    coefs = {}
    for x in x_vars:
        if x not in all_list:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}
            continue
        xi   = all_list.index(x)
        cols = [l * k_vars + xi for l in range(n_short)
                if l * k_vars + xi < gamma.shape[1]]
        if cols:
            c = float(gamma[y_idx, cols].sum())
            if gamma_se is not None:
                se = float(np.sqrt(np.sum(gamma_se[y_idx, cols] ** 2)))
                t  = c / (se + 1e-12)
                pv = float(t_dist.sf(abs(t), df=max(1, res.nobs - k_vars)) * 2)
            else:
                pv = np.nan
        else:
            c = pv = np.nan
        coefs[x] = {'coef': c, 'pval': pv}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'coefs': coefs,
            'n_coint': n_ci, 'nobs': int(res.nobs)}


print('Funciones de extraccion listas (ARDL usa metodologia BJ si esta disponible).')


## 8. Estimación Iterativa

In [ ]:
# Generar TODAS las combinaciones de TODOS los tamaños ────────────────────────
_n_max = len(X_SAFE) if MAX_VARS_POR_MODELO is None else MAX_VARS_POR_MODELO
_combos = []
for _r in range(1, _n_max + 1):
    _combos.extend(list(combinations(X_SAFE, _r)))

if MAX_COMBINACIONES is not None and len(_combos) > MAX_COMBINACIONES:
    random.seed(SEMILLA)
    _combos = random.sample(_combos, MAX_COMBINACIONES)
    print(f'Muestreadas aleatoriamente: {MAX_COMBINACIONES} de {len(_combos)}')

_n_runs = len(_combos) * len(CRITERIOS_SEL) * (CORRER_ARDL + CORRER_VAR) + len(_combos) * CORRER_VECM
print(f'Combinaciones: {len(_combos)} | Criterios: {CRITERIOS_SEL}')
print(f'Estimaciones totales ≈ {_n_runs}\n')

RESULTADOS = []
_t0 = datetime.now()

# ── ARDL y VAR: loop por criterio de selección de rezagos ────────────────────
for _criterio in CRITERIOS_SEL:
    for _ci, _combo in enumerate(_combos):
        _xv = list(_combo)
        _tv = [Y_SAFE] + _xv
        _df = df_safe[_tv].dropna()
        if len(_df) < 30:
            continue

        _base = {
            'combo_id':    f'{_ci}_{_criterio}',
            'criterio':    _criterio,
            'n_vars':      len(_xv),
            'vars_safe':   '|'.join(_xv),
            'vars_label':  '|'.join([COL_LABEL[v] for v in _xv]),
        }

        if CORRER_ARDL:
            try:
                _r = extraer_ardl(_df, Y_SAFE, _xv, MAX_LAGS_DEP, MAX_LAGS_IND, _criterio)
                _row = {**_base, 'modelo': 'ARDL',
                        'aic': _r['aic'], 'bic': _r['bic'], 'hq': _r['hq'], 'nobs': _r['nobs']}
                for _x in _xv:
                    _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                    _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
                RESULTADOS.append(_row)
            except Exception:
                pass

        if CORRER_VAR:
            try:
                _r = extraer_var(_df, Y_SAFE, _xv, _tv, MAX_LAGS_VAR, _criterio)
                _row = {**_base, 'modelo': 'VAR',
                        'aic': _r['aic'], 'bic': _r['bic'], 'hq': _r['hq'], 'nobs': _r['nobs']}
                for _x in _xv:
                    _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                    _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
                RESULTADOS.append(_row)
            except Exception:
                pass

# ── VECM: una vez por combinación (no depende del criterio de selección) ──────
if CORRER_VECM:
    for _ci, _combo in enumerate(_combos):
        _xv = list(_combo)
        _tv = [Y_SAFE] + _xv
        _df = df_safe[_tv].dropna()
        if len(_df) < 30:
            continue
        _base = {
            'combo_id':    f'{_ci}_vecm',
            'criterio':    'johansen',
            'n_vars':      len(_xv),
            'vars_safe':   '|'.join(_xv),
            'vars_label':  '|'.join([COL_LABEL[v] for v in _xv]),
        }
        try:
            _r = extraer_vecm(_df, Y_SAFE, _xv, _tv, k_ar_diff=2, det_order=0)
            if _r is not None:
                _row = {**_base, 'modelo': 'VECM',
                        'aic': _r['aic'], 'bic': _r['bic'], 'hq': _r['hq'],
                        'nobs': _r['nobs'], 'n_coint': _r['n_coint']}
                for _x in _xv:
                    _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                    _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
                RESULTADOS.append(_row)
        except Exception:
            pass

        if (_ci + 1) % 20 == 0 or _ci == len(_combos) - 1:
            _elapsed = (datetime.now() - _t0).seconds
            print(f'  VECM {_ci+1:>3}/{len(_combos)} — total acumulado: {len(RESULTADOS):>5}  [{_elapsed}s]')

df_res = pd.DataFrame(RESULTADOS)
print(f'\n✅ Estimación completa: {len(df_res)} filas')
print(df_res.groupby(['modelo','criterio']).size().to_string())

## 9. Resultados en Formato Largo

In [ ]:
# Transformar a formato largo: una fila por (combo × modelo × variable_X)
_filas = []
for _, _fila in df_res.iterrows():
    _xv = [v for v in _fila['vars_safe'].split('|') if v]
    for _x in _xv:
        _coef = _fila.get(f'coef_{_x}', np.nan)
        _pval = _fila.get(f'pval_{_x}', np.nan)
        if pd.isna(_coef):
            continue
        _filas.append({
            'combo_id':   _fila['combo_id'],
            'modelo':     _fila['modelo'],
            'var_safe':   _x,
            'variable':   COL_LABEL.get(_x, _x),
            'coef':       _coef,
            'pval':       _pval,
            'sig':        (_pval < NIVEL_SIG) if not pd.isna(_pval) else False,
            'aic':        _fila['aic'],
            'bic':        _fila['bic'],
            'hq':         _fila['hq'],
            'n_vars':     _fila['n_vars'],
            'vars_label': _fila['vars_label'],
        })

df_largo = pd.DataFrame(_filas)
print(f'DataFrame largo: {df_largo.shape[0]} filas × {df_largo.shape[1]} cols')
print(f'Variables con datos: {df_largo["variable"].nunique()}')
print(f'Modelos: {df_largo["modelo"].value_counts().to_dict()}')
display(df_largo.head(8))

## 10. Scatter: Coeficiente vs P-valor por Variable

In [ ]:
_COLORS = {'ARDL': '#2196F3', 'VAR': '#FF9800', 'VECM': '#4CAF50'}

_vars_plot = [v for v in [COL_LABEL[x] for x in X_SAFE]
              if v in df_largo['variable'].unique()]

if not _vars_plot:
    print('⚠️ Sin datos para graficar.')
else:
    _nc = min(2, len(_vars_plot))
    _nr = (len(_vars_plot) + _nc - 1) // _nc

    fig = make_subplots(
        rows=_nr, cols=_nc,
        subplot_titles=[f'<b>{v[:55]}</b>' for v in _vars_plot],
        horizontal_spacing=0.10,
        vertical_spacing=0.10,
    )

    _legend_added = set()
    for _idx, _var in enumerate(_vars_plot):
        _row = _idx // _nc + 1
        _col = _idx % _nc + 1

        _dv = df_largo[df_largo['variable'] == _var].dropna(subset=['coef', 'pval'])
        if _dv.empty:
            continue

        for _mod, _color in _COLORS.items():
            _dm = _dv[_dv['modelo'] == _mod]
            if _dm.empty:
                continue

            # Tamaño del punto inversamente proporcional al AIC (mejor modelo = punto mayor)
            _aic_v = _dm['aic']
            _amin, _amax = _aic_v.min(), _aic_v.max()
            _sz = (8 + 12 * (1 - (_aic_v - _amin) / (_amax - _amin + 1e-9))).tolist()

            _hover = _dm.apply(
                lambda r: (
                    f"Coef: {r['coef']:.4f}<br>"
                    f"P-val: {r['pval']:.4f}<br>"
                    f"AIC: {r['aic']:.2f}<br>"
                    f"Vars: {str(r['vars_label'])[:60]}"
                ), axis=1
            )

            _sym = ['circle' if s else 'x' for s in _dm['sig']]
            _show = _mod not in _legend_added
            _legend_added.add(_mod)

            fig.add_trace(
                go.Scatter(
                    x=_dm['coef'], y=_dm['pval'],
                    mode='markers',
                    name=_mod,
                    showlegend=_show,
                    legendgroup=_mod,
                    marker=dict(
                        color=_color, size=_sz, opacity=0.75,
                        symbol=_sym,
                        line=dict(color='white', width=0.8)
                    ),
                    hovertext=_hover, hoverinfo='text'
                ),
                row=_row, col=_col
            )

        # Líneas de referencia
        fig.add_hline(y=NIVEL_SIG, line=dict(color='red', dash='dot', width=1),
                      row=_row, col=_col)
        fig.add_vline(x=0, line=dict(color='gray', dash='dot', width=1),
                      row=_row, col=_col)
        fig.update_yaxes(autorange='reversed', title_text='P-valor ↑mejor',
                         row=_row, col=_col)
        fig.update_xaxes(title_text='Coeficiente (suma lags)',
                         row=_row, col=_col)

    fig.update_layout(
        height=420 * _nr,
        template='plotly_white',
        title=(
            'Coeficiente vs P-valor por variable explicativa'
            '<br><sup>● = sig (p<0.05) | ✗ = no sig | tamaño ∝ calidad (AIC)</sup>'
        ),
    )
    fig.show()

## 11. Criterios de Información: AIC · BIC · HQ

In [ ]:
# ── Distribución por tipo de modelo (box plots) ───────────────────────────────
fig_box = make_subplots(rows=1, cols=3,
                        subplot_titles=['AIC', 'BIC', 'HQ (Hannan-Quinn)'])
for _ci, _crit in enumerate(['aic', 'bic', 'hq']):
    for _mod, _color in _COLORS.items():
        _dv = df_res[df_res['modelo'] == _mod][_crit].dropna()
        if _dv.empty:
            continue
        fig_box.add_trace(
            go.Box(y=_dv, name=_mod, marker_color=_color,
                   showlegend=(_ci == 0), legendgroup=_mod,
                   boxpoints='outliers', jitter=0.3),
            row=1, col=_ci+1
        )
fig_box.update_layout(height=420, template='plotly_white',
                      title='Distribución de criterios de información por modelo')
fig_box.show()

# ── Top combinaciones por AIC ─────────────────────────────────────────────────
_top_n = 20
_best  = (df_res.groupby(['combo_id', 'modelo'])[['aic', 'vars_label']]
          .first().reset_index()
          .sort_values('aic').head(_top_n * 3))

fig_top = go.Figure()
for _mod, _color in _COLORS.items():
    _dm = _best[_best['modelo'] == _mod].head(_top_n)
    if _dm.empty:
        continue
    fig_top.add_trace(go.Bar(
        x=[f'C{int(r["combo_id"])}' for _, r in _dm.iterrows()],
        y=_dm['aic'],
        name=_mod,
        marker_color=_color,
        opacity=0.85,
        hovertext=_dm['vars_label'].str[:80],
        hoverinfo='x+y+text'
    ))
fig_top.update_layout(
    barmode='group', template='plotly_white',
    title=f'AIC — Top {_top_n} combinaciones (menor = mejor)',
    xaxis_title='Combinación', yaxis_title='AIC', height=450
)
fig_top.show()

# ── Evolución AIC por número de variables ─────────────────────────────────────
fig_nv = go.Figure()
for _mod, _color in _COLORS.items():
    _dm = df_res[df_res['modelo'] == _mod].groupby('n_vars')['aic'].agg(['mean', 'min', 'max'])
    if _dm.empty:
        continue
    fig_nv.add_trace(go.Scatter(
        x=_dm.index, y=_dm['mean'],
        mode='lines+markers',
        name=_mod + ' (media)',
        line=dict(color=_color, width=2),
        marker=dict(size=8)
    ))
    fig_nv.add_trace(go.Scatter(
        x=_dm.index, y=_dm['min'],
        mode='lines',
        name=_mod + ' (min)',
        line=dict(color=_color, width=1, dash='dot'),
        showlegend=False
    ))
fig_nv.update_layout(
    template='plotly_white',
    title='AIC medio y mínimo por cantidad de variables explicativas',
    xaxis_title='N° vars explicativas', yaxis_title='AIC', height=400
)
fig_nv.show()

## 12. Tabla Resumen y Ranking de Variables

In [ ]:
# ── Mejor modelo por tipo ─────────────────────────────────────────────────────
print('═'*70)
print('MEJOR MODELO POR TIPO (mínimo AIC)')
print('═'*70)
for _mod in ['ARDL', 'VAR', 'VECM']:
    _dm = df_res[df_res['modelo'] == _mod]
    if _dm.empty:
        continue
    _best_row = _dm.loc[_dm['aic'].idxmin()]
    _xv_safe  = [v for v in _best_row['vars_safe'].split('|') if v]
    _xv_label = [COL_LABEL.get(v, v) for v in _xv_safe]
    print(f'\n■ {_mod}')
    print(f'  AIC={_best_row["aic"]:.4f}  BIC={_best_row["bic"]:.4f}  HQ={_best_row["hq"]:.4f}')
    print(f'  Obs={int(_best_row["nobs"])}  N_vars={int(_best_row["n_vars"])}')
    print(f'  Variables: {", ".join(_xv_label)}')
    print(f'  Coeficientes:')
    for _x, _xl in zip(_xv_safe, _xv_label):
        _c = _best_row.get(f'coef_{_x}', np.nan)
        _p = _best_row.get(f'pval_{_x}', np.nan)
        if pd.isna(_c):
            continue
        _sig = '***' if _p < 0.01 else '**' if _p < 0.05 else '*' if _p < 0.10 else ''
        print(f'    {_xl[:50]:<50}  coef={_c:+.4f}  p={_p:.4f} {_sig}')
print('\n  * p<10%  ** p<5%  *** p<1%')

# ── Frecuencia de significancia por variable ──────────────────────────────────
print('\n' + '═'*70)
print('FRECUENCIA DE SIGNIFICANCIA (p < 5%) POR VARIABLE')
print('═'*70)
if not df_largo.empty:
    _sig_cnt = (
        df_largo[df_largo['sig']]
        .groupby(['variable', 'modelo']).size()
        .unstack(fill_value=0)
        .assign(Total=lambda x: x.sum(axis=1))
        .sort_values('Total', ascending=False)
    )
    _total_combos = df_largo.groupby('modelo')['combo_id'].nunique()
    print('(Sobre el total de combinaciones donde aparece cada variable)')
    display(_sig_cnt)

# ── Mediana de p-valor por variable ──────────────────────────────────────────
print('\n' + '═'*70)
print('MEDIANA DE P-VALOR POR VARIABLE Y MODELO (menor = más robusto)')
print('═'*70)
if not df_largo.empty:
    _pmed = (
        df_largo.groupby(['variable', 'modelo'])['pval']
        .median().unstack().round(4)
        .sort_values(df_largo['modelo'].iloc[0] if len(df_largo) else 'ARDL')
    )
    display(_pmed)

# ── Tabla completa exportable ─────────────────────────────────────────────────
print('\n' + '═'*70)
print('RESULTADOS COMPLETOS (guardados en Variables Regresivas/resultados_iterativos.csv)')
print('═'*70)
_out = df_largo[['modelo','variable','coef','pval','sig','aic','bic','hq','n_vars','vars_label']].copy()
_out = _out.sort_values(['modelo','variable','pval'])
_out_path = Path('Variables Regresivas') / 'resultados_iterativos.csv'
_out.to_csv(_out_path, index=False)
print(f'  Guardado en: {_out_path}')
display(_out.head(20))